### Bronze Layer

In [0]:
%sql
select * from parquet.`s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/raw-data`
limit 2

-- 43,05,006

In [0]:
%sql
create catalog if not exists dev

In [0]:
%sql
create schema if not exists dev.taxi_db

### Ingest into bronze table

In [0]:
source_s3_path = "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/raw-data/"
checkpoints_bronze = "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/checkpoints/bronze/"


# Extract from S3 & Load to Bronze Delta table

(spark.readStream
 .format("cloudFiles")
 .option('cloudFiles.format', 'parquet')
 .option('cloudFiles.schemaLocation', checkpoints_bronze)
 .option('cloudFiles.schemaEvolutionMode','addNewColumnsWithTypeWidening')
 .load(source_s3_path)
 .withColumn('ingested_at', F.current_timestamp())
 .writeStream
 .option('mergeSchema', 'true')
 .option('checkpointLocation', checkpoints_bronze)
 .trigger(availableNow=True)
 .toTable("dev.taxi_db.bronze_yellow_taxi")

)


# .option("badRecordsPath", "s3://yellow-taxi-raw-parquet-806121397587-ap-south-2-an/bad-records/")

In [0]:
%sql
select *
from dev.taxi_db.bronze_yellow_taxi
limit 2

